In [1]:
import pandas as pd


In [2]:
df = pd.read_csv(r'D:\DBI\Final progect\Brazilian E-Commerce Public Dataset by Olist\Olist_Master_Dataset.csv')
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,seller_city,seller_state,customer_lat,customer_lng,geolocation_city,geolocation_state,seller_lat,seller_lng,geolocation_city_seller_geo,geolocation_state_seller_geo
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,maua,SP,-23.574809,-46.587471,sao paulo,SP,-23.680114,-46.452454,maua,SP
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,maua,SP,-23.574809,-46.587471,sao paulo,SP,-23.680114,-46.452454,maua,SP
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,maua,SP,-23.574809,-46.587471,sao paulo,SP,-23.680114,-46.452454,maua,SP
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,...,belo horizonte,SP,-12.169860,-44.988369,barreiras,BA,-19.810119,-43.984727,belo horizonte,MG
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,...,guariba,SP,-16.746337,-48.514624,vianopolis,GO,-21.362358,-48.232976,guariba,SP


In [3]:
date_cols = ['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date', 
             'order_estimated_delivery_date']

In [4]:
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')



In [5]:


df.dropna(subset=['order_delivered_customer_date'], inplace=True)

if 'review_score' in df.columns:
    df['review_score'] = df['review_score'].fillna(df['review_score'].median())

if 'product_category_name_english' in df.columns:
    df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')

In [6]:
text_cols = ['customer_city', 'customer_state', 'product_category_name_english']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().str.strip()

In [7]:
q99 = df['price'].quantile(0.99)
df_clean = df[df['price'] <= q99].copy()

In [8]:
df_clean.drop_duplicates(inplace=True)

In [9]:
import pandas as pd
import plotly.express as px

In [10]:
df_clean['year_month'] = df_clean['order_purchase_timestamp'].dt.to_period('M').astype(str)

In [11]:
monthly_sales = df_clean.groupby('year_month')['price'].sum().reset_index()

fig_trend = px.line(monthly_sales, 
                    x='year_month', 
                    y='price', 
                    title='Monthly Sales Trend Overview',
                    labels={'year_month': 'Timeline (Year-Month)', 'price': 'Total Revenue'},
                    markers=True)

fig_trend.update_layout(template='plotly_dark', hovermode='x unified')
fig_trend.show()

In [12]:
# --- Dashboard 2: Top 10 Product Categories ---
top_categories = df_clean.groupby('product_category_name_english')['price'].sum().reset_index()
top_categories = top_categories.sort_values(by='price', ascending=False).head(10)

fig_cats = px.bar(top_categories, 
                  x='price', 
                  y='product_category_name_english', 
                  orientation='h',
                  title='Top 10 Performing Product Categories',
                  labels={'price': 'Total Revenue', 'product_category_name_english': 'Product Category'},
                  color='price',
                  color_continuous_scale='Blues')

fig_cats.update_layout(template='plotly_dark', yaxis={'categoryorder':'total ascending'})
fig_cats.show()

In [13]:

layout_style = 'plotly_dark'

if 'payment_type' in df_clean.columns:
    pay_dist = df_clean['payment_type'].value_counts().reset_index()
    pay_dist.columns = ['payment_type', 'count']
    fig3 = px.pie(pay_dist, names='payment_type', values='count', hole=0.4,
                  title='3. Customer Payment Methods Preference')
    fig3.update_layout(template=layout_style)
    fig3.show()

In [14]:
df_clean['day_of_week'] = df_clean['order_purchase_timestamp'].dt.day_name()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_sales = df_clean.groupby('day_of_week')['order_id'].count().reset_index()

fig4 = px.bar(day_sales, x='day_of_week', y='order_id', 
              title='4. Order Volume by Day of the Week',
              category_orders={'day_of_week': days_order}, 
              color='order_id', color_continuous_scale='Purples')

fig4.update_layout(template=layout_style)
fig4.show()

In [15]:
if 'customer_state' in df_clean.columns:
    state_sales = df_clean.groupby('customer_state')['price'].sum().reset_index()
    state_sales = state_sales.sort_values(by='price', ascending=False).head(10)
    fig5 = px.bar(state_sales, x='customer_state', y='price',
                  title='5. Top 10 Customer States by Total Revenue', color='price', color_continuous_scale='Teal')
    fig5.update_layout(template=layout_style)
    fig5.show()

In [16]:
df_clean['delivery_days'] = (df_clean['order_delivered_customer_date'] - df_clean['order_purchase_timestamp']).dt.days
fig6 = px.histogram(df_clean[df_clean['delivery_days'] <= 40], x='delivery_days', nbins=40,
                    title='6. Delivery Time Distribution (in Days)')
fig6.update_layout(template=layout_style, xaxis_title='Days to Deliver', yaxis_title='Number of Orders')
fig6.show()

In [17]:
top_cats = df_clean.groupby('product_category_name_english')['price'].sum().reset_index()
top_cats = top_cats.sort_values(by='price', ascending=False).head(10)
if 'review_score' in df_clean.columns:
    top_cat_names = top_cats['product_category_name_english'].tolist()
    reviews = df_clean[df_clean['product_category_name_english'].isin(top_cat_names)]
    avg_reviews = reviews.groupby('product_category_name_english')['review_score'].mean().reset_index()
    fig7 = px.bar(avg_reviews, x='review_score', y='product_category_name_english', orientation='h',
                  title='7. Average Customer Satisfaction (Top Categories)', range_x=[0, 5])
    fig7.update_layout(template=layout_style, yaxis={'categoryorder':'total ascending'})
    fig7.show()

In [18]:
cat_metrics = df_clean.groupby('product_category_name_english').agg(
    avg_price=('price', 'mean'), avg_freight=('freight_value', 'mean'), total_orders=('order_id', 'count')
).reset_index()
cat_metrics = cat_metrics[cat_metrics['total_orders'] > 50] # Filter small categories
fig8 = px.scatter(cat_metrics, x='avg_price', y='avg_freight', size='total_orders', 
                  hover_name='product_category_name_english',
                  title='8. Relationship: Avg Item Price vs. Avg Freight Value per Category',
                  color='avg_price', color_continuous_scale='Magma')
fig8.update_layout(template=layout_style)
fig8.show()

In [19]:
%pip install --user xgboost scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [20]:
import pandas as pd
import numpy as np
import plotly.express as px
# 2. Data Preprocessing & Dimensionality Reduction (PCA)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.decomposition import PCA


from sklearn.linear_model import LinearRegression

from sklearn.svm import SVC, SVR

from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.naive_bayes import GaussianNB

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor

from sklearn.cluster import KMeans, DBSCAN

from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             mean_squared_error, mean_absolute_error, r2_score)

print("All Machine Learning libraries imported successfully!")

All Machine Learning libraries imported successfully!


In [31]:
latest_date = df_clean['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = df_clean.groupby('customer_id').agg({
    'order_purchase_timestamp': lambda x: (latest_date - x.max()).days,
    'order_id': 'nunique',
    'price': 'sum'
}).reset_index()

rfm.columns = ['customer_id', 'Recency', 'Frequency', 'Monetary']

rfm = rfm[(rfm['Monetary'] <= rfm['Monetary'].quantile(0.99)) & 
          (rfm['Frequency'] <= rfm['Frequency'].quantile(0.99))]

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

cluster_mapping = {
    0: 'Potential/New Customers',
    1: 'At-Risk/Churned Customers',
    2: 'Loyal Customers',
    3: 'VIP/High-Spenders'
}
rfm['Customer_Segment'] = rfm['Cluster'].map(cluster_mapping)

print("Clustering completed successfully!")
print(rfm['Customer_Segment'].value_counts())

layout_style = 'plotly_dark'

fig_kmeans = px.scatter_3d(rfm, 
                           x='Recency', 
                           y='Frequency', 
                           z='Monetary',
                           color='Customer_Segment',
                           title='Customer Segmentation (3D K-Means Clustering)',
                           opacity=0.7,
                           color_discrete_sequence=px.colors.qualitative.Pastel)

fig_kmeans.update_layout(template=layout_style, 
                         scene=dict(xaxis_title='Recency (Days)',
                                    yaxis_title='Frequency (Orders)',
                                    zaxis_title='Monetary (Revenue)'))
fig_kmeans.show()

Clustering completed successfully!
Customer_Segment
VIP/High-Spenders            32404
Potential/New Customers      32317
Loyal Customers              20863
At-Risk/Churned Customers     8893
Name: count, dtype: int64


In [32]:
df_clean['is_late'] = (df_clean['order_delivered_customer_date'] > df_clean['order_estimated_delivery_date']).astype(int)

df_clean['promised_delivery_days'] = (df_clean['order_estimated_delivery_date'] - df_clean['order_purchase_timestamp']).dt.days

features = ['price', 'freight_value', 'promised_delivery_days']
X = df_clean[features].dropna()
y = df_clean.loc[X.index, 'is_late']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training the Random Forest model (this might take a few seconds)...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("\nModel Evaluation (Classification Report):")
print(classification_report(y_test, y_pred))

layout_style = 'plotly_dark'
cm = confusion_matrix(y_test, y_pred)

fig_cm = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                   labels=dict(x="Predicted Label", y="True Label", color="Count"),
                   x=['On Time (0)', 'Late (1)'], y=['On Time (0)', 'Late (1)'],
                   title='Confusion Matrix: Late Delivery Prediction (Random Forest)')

fig_cm.update_layout(template=layout_style)
fig_cm.show()

Training the Random Forest model (this might take a few seconds)...

Model Evaluation (Classification Report):
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     21156
           1       0.71      0.01      0.02      1757

    accuracy                           0.92     22913
   macro avg       0.82      0.51      0.49     22913
weighted avg       0.91      0.92      0.89     22913



In [37]:
import plotly.graph_objects as go

df_clean['order_purchase_timestamp'] = pd.to_datetime(df_clean['order_purchase_timestamp'])
df_clean['purchase_date'] = df_clean['order_purchase_timestamp'].dt.date

daily_sales = df_clean.groupby('purchase_date')['price'].sum().reset_index()
daily_sales.columns = ['date', 'total_sales']
daily_sales['date'] = pd.to_datetime(daily_sales['date'])

daily_sales = daily_sales.sort_values('date')

daily_sales['day_of_week'] = daily_sales['date'].dt.dayofweek
daily_sales['day_of_month'] = daily_sales['date'].dt.day
daily_sales['month'] = daily_sales['date'].dt.month
daily_sales['year'] = daily_sales['date'].dt.year

X = daily_sales[['day_of_week', 'day_of_month', 'month', 'year']]
y = daily_sales['total_sales']

split_idx = int(len(daily_sales) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = daily_sales['date'].iloc[split_idx:]

print("Training XGBoost Regressor for Sales Forecasting...")
xgb_model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
print("\nModel Evaluation:")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")

layout_style = 'plotly_dark'
fig_forecast = go.Figure()

fig_forecast.add_trace(go.Scatter(x=dates_test, y=y_test, mode='lines', 
                                  name='Actual Sales', line=dict(color='cyan')))
fig_forecast.add_trace(go.Scatter(x=dates_test, y=y_pred, mode='lines', 
                                  name='Predicted Sales', line=dict(color='orange', dash='dash')))

fig_forecast.update_layout(title='Sales Forecasting: Actual vs Predicted (Testing Period)',
                           xaxis_title='Date',
                           yaxis_title='Total Sales Revenue',
                           template=layout_style,
                           hovermode="x unified")
fig_forecast.show()

Training XGBoost Regressor for Sales Forecasting...

Model Evaluation:
Mean Absolute Error (MAE): $6948.37


In [34]:
df_clean['delivery_days'] = (pd.to_datetime(df_clean['order_delivered_customer_date']) - 
                             pd.to_datetime(df_clean['order_purchase_timestamp'])).dt.days

df_clean['estimated_days'] = (pd.to_datetime(df_clean['order_estimated_delivery_date']) - 
                              pd.to_datetime(df_clean['order_purchase_timestamp'])).dt.days

features = ['price', 'freight_value', 'delivery_days', 'estimated_days']

df_subset = df_clean.dropna(subset=features + ['review_score'])

X = df_subset[features]
y = df_subset['review_score'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Decision Tree Classifier...")
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced')
dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)

print("\nClassification Report (1 to 5 Stars):")
print(classification_report(y_test, y_pred, zero_division=0))

layout_style = 'plotly_dark'
importance = dt_model.feature_importances_

fig_imp = px.bar(x=importance, y=features, orientation='h',
                 title='Feature Importance: What drives customer satisfaction?',
                 labels={'x': 'Impact Level (Importance)', 'y': 'Store Metric'},
                 color=importance, color_continuous_scale='Reds')

fig_imp.update_layout(template=layout_style, yaxis={'categoryorder':'total ascending'})
fig_imp.show()

Training Decision Tree Classifier...

Classification Report (1 to 5 Stars):
              precision    recall  f1-score   support

           1       0.51      0.29      0.37      2556
           2       0.08      0.07      0.07       727
           3       0.11      0.18      0.14      2014
           4       0.21      0.40      0.27      4338
           5       0.66      0.46      0.54     13278

    accuracy                           0.39     22913
   macro avg       0.31      0.28      0.28     22913
weighted avg       0.49      0.39      0.42     22913



In [36]:
latest_date = df_clean['order_purchase_timestamp'].max()

churn_data = df_clean.groupby('customer_id').agg(
    Recency=('order_purchase_timestamp', lambda x: (latest_date - x.max()).days),
    Monetary=('price', 'sum'),
    Avg_Review=('review_score', 'mean'),
    Avg_Freight=('freight_value', 'mean'),
    Avg_Delivery=('delivery_days', 'mean')
).reset_index()

churn_data['Churn'] = (churn_data['Recency'] > 180).astype(int)

features = ['Monetary', 'Avg_Review', 'Avg_Freight', 'Avg_Delivery']
X = churn_data[features].dropna()
y = churn_data.loc[X.index, 'Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training XGBoost Classifier...")
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

print("\nModel Evaluation (Churn Prediction - 0: Active, 1: Churned):")
print(classification_report(y_test, y_pred, zero_division=0))

layout_style = 'plotly_dark'
importance = xgb_model.feature_importances_

fig_churn = px.bar(x=importance, y=features, orientation='h',
                   title='Factors Influencing Customer Churn (XGBoost)',
                   labels={'x': 'Impact Level (Importance)', 'y': 'Customer Metric'},
                   color=importance, color_continuous_scale='Viridis')

fig_churn.update_layout(template=layout_style, yaxis={'categoryorder':'total ascending'})
fig_churn.show()

Training XGBoost Classifier...

Model Evaluation (Churn Prediction - 0: Active, 1: Churned):
              precision    recall  f1-score   support

           0       0.80      0.65      0.72      7709
           1       0.79      0.89      0.84     11377

    accuracy                           0.79     19086
   macro avg       0.80      0.77      0.78     19086
weighted avg       0.80      0.79      0.79     19086



In [35]:
clv_data = df_clean.groupby('customer_id').agg(
    Frequency=('order_id', 'nunique'),
    Avg_Freight=('freight_value', 'mean'),
    Avg_Delivery=('delivery_days', 'mean'),
    Monetary=('price', 'sum')
).reset_index()

features = ['Frequency', 'Avg_Freight', 'Avg_Delivery']
X = clv_data[features].dropna()
y = clv_data.loc[X.index, 'Monetary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Linear Regression Model...")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"\nModel Evaluation:")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"R-squared Score (R2): {r2:.2f}")

layout_style = 'plotly_dark'

fig_clv = px.scatter(x=y_test, y=y_pred, 
                     labels={'x': 'Actual Customer Value ($)', 'y': 'Predicted Customer Value ($)'},
                     title='Customer Lifetime Value Prediction (Linear Regression)',
                     opacity=0.6, color_discrete_sequence=['cyan'])

max_val = max(y_test.max(), y_pred.max())
fig_clv.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val], mode='lines', 
                             name='Perfect Prediction', line=dict(color='orange', dash='dash')))

fig_clv.update_layout(template=layout_style, hovermode="closest")
fig_clv.show()

Training Linear Regression Model...

Model Evaluation:
Mean Absolute Error (MAE): $81.48
R-squared Score (R2): 0.09


In [38]:
from sklearn.neighbors import NearestNeighbors

category_data = df_clean.groupby('product_category_name_english').agg(
    Avg_Price=('price', 'mean'),
    Avg_Freight=('freight_value', 'mean'),
    Total_Orders=('order_id', 'count'),
    Avg_Review=('review_score', 'mean')
).reset_index()

category_data = category_data[category_data['Total_Orders'] > 50].reset_index(drop=True)

features = ['Avg_Price', 'Avg_Freight', 'Total_Orders', 'Avg_Review']
X_cats = category_data[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cats)

print("Training KNN Model for Recommendations...")
knn = NearestNeighbors(n_neighbors=4, metric='euclidean')
knn.fit(X_scaled)

distances, indices = knn.kneighbors(X_scaled)

test_category = 'health_beauty'
if test_category in category_data['product_category_name_english'].values:
    cat_idx = category_data[category_data['product_category_name_english'] == test_category].index[0]
    
    neighbor_indices = indices[cat_idx][1:]
    recommended_cats = category_data.iloc[neighbor_indices]['product_category_name_english'].tolist()
    
    print(f"\n--- [Recommendation Engine Demo] ---")
    print(f"If a customer is viewing: '{test_category.upper()}'")
    print(f"The KNN model suggests cross-selling: {recommended_cats}")
    print(f"------------------------------------\n")

layout_style = 'plotly_dark'
top_n = 20 
fig_knn = go.Figure()

sample_cats = category_data.sort_values('Total_Orders', ascending=False).head(top_n)

fig_knn.add_trace(go.Scatter(
    x=sample_cats['Avg_Price'], 
    y=sample_cats['Avg_Review'],
    mode='markers+text',
    text=sample_cats['product_category_name_english'],
    textposition="top center",
    marker=dict(size=sample_cats['Total_Orders']/200, color='cyan', opacity=0.7),
    name='Product Categories'
))

fig_knn.update_layout(
    title='Product Category Similarity Matrix (KNN Feature Space)',
    xaxis_title='Average Price ($)',
    yaxis_title='Average Review Score (Stars)',
    template=layout_style,
    hovermode="closest"
)
fig_knn.show()

Training KNN Model for Recommendations...

--- [Recommendation Engine Demo] ---
If a customer is viewing: 'HEALTH_BEAUTY'
The KNN model suggests cross-selling: ['sports_leisure', 'housewares', 'computers_accessories']
------------------------------------



In [39]:
physical_features = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

available_features = [col for col in physical_features if col in df_clean.columns]

if len(available_features) > 1:
    print(f"Applying PCA on physical features: {available_features}")
    df_freight = df_clean.dropna(subset=available_features + ['freight_value', 'price']).copy()
    
    scaler = StandardScaler()
    scaled_dims = scaler.fit_transform(df_freight[available_features])
    
    pca = PCA(n_components=1, random_state=42)
    df_freight['Size_Index'] = pca.fit_transform(scaled_dims)
    
    X = df_freight[['Size_Index', 'price']]
else:
    print("Physical features missing. Using 'price' and order counts as proxy features.")
    df_freight = df_clean.dropna(subset=['price', 'freight_value']).copy()
    X = df_freight[['price']] 

y = df_freight['freight_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Multiple Regression Model...")
multi_reg = LinearRegression()
multi_reg.fit(X_train, y_train)

y_pred = multi_reg.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Evaluation (Predicting Freight Value):")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"R-squared Score (R2): {r2:.2f}")

layout_style = 'plotly_dark'

plot_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred}).sample(n=min(1000, len(y_test)), random_state=42)

fig_freight = px.scatter(plot_df, x='Actual', y='Predicted', 
                         title='8. Freight Value Prediction (PCA + Multi Regression)',
                         labels={'Actual': 'Actual Freight Cost ($)', 'Predicted': 'Predicted Freight Cost ($)'},
                         opacity=0.6, color_discrete_sequence=['#00CC96'])

max_val = max(plot_df['Actual'].max(), plot_df['Predicted'].max())
fig_freight.add_shape(type="line", x0=0, y0=0, x1=max_val, y1=max_val,
                      line=dict(color="orange", width=2, dash="dash"))

fig_freight.update_layout(template=layout_style, hovermode="closest")
fig_freight.show()

Applying PCA on physical features: ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
Training Multiple Regression Model...

Model Evaluation (Predicting Freight Value):
Mean Absolute Error (MAE): $7.13
R-squared Score (R2): 0.33


In [29]:
import joblib

# Exporting models to disk
joblib.dump(rf_model, 'rf_late_delivery_model.pkl')
joblib.dump(xgb_model, 'xgb_churn_model.pkl')

print("Models saved successfully!")

Models saved successfully!


In [30]:
df_clean.to_csv('cleaned_data.csv', index=False)